## Sesion 1 - Control de versiones y estructura de proyecto

**Bootcamp:** Python PY4  
**Modulo 4:** Big Data y MLOps  
**Taller de Calidad: Pruebas Unitarias y Documentación Profesional**

---

Este semana nos enfocamos en hacer que el codigo sea **confiable, reproducible y mantenible**. Esa es la diferencia entre un script que uno escribe para si mismo y un proyecto de ingenieria que otras personas (o uno mismo, seis meses despues) pueden retomar sin problemas.

Un analisis que nadie mas puede reproducir, o que se rompe apenas alguien mas lo ejecuta en su computadora, no tiene mucho valor en un entorno profesional. En esta sesion vamos a transformar un script suelto en un proyecto con estructura profesional, usando Git para el control de versiones.

## Objetivos de la sesion

Al final de esta sesion vas a poder:

- Explicar por que la estructura de un proyecto y el control de versiones importan en ciencia de datos.
- Organizar un proyecto siguiendo el estandar de carpetas `data/`, `src/`, `tests/`.
- Inicializar un repositorio Git y hacer commits desde cero.
- Configurar un archivo `.gitignore` para evitar subir archivos pesados o sensibles.
- Escribir un `requirements.txt` para que el proyecto sea reproducible por otra persona.

## Parte 0: Preparacion del entorno

Antes de empezar, un detalle importante sobre Colab: cada vez que se cierra o reinicia el entorno, se pierde todo lo que no este guardado en Google Drive. Como este proyecto lo vamos a construir a lo largo de tres sesiones distintas, necesitamos que la carpeta del proyecto (junto con su historial de Git) persista de una sesion a otra.

Por eso vamos a trabajar directamente sobre una carpeta dentro de Google Drive. Al iniciar cada sesion, lo primero que haremos sera montar Drive y ubicarnos en esa misma carpeta.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Ahora definimos la ruta del proyecto. Vamos a usar siempre esta misma variable a lo largo de las tres sesiones, asi que si necesitas cambiar el nombre de la carpeta, este es el unico lugar donde hay que hacerlo.

In [3]:
import os

PROJECT_NAME = "taller_calidad"
PROJECT_PATH = f"/content/drive/MyDrive/{PROJECT_NAME}"

os.makedirs(PROJECT_PATH, exist_ok=True)
os.chdir(PROJECT_PATH)

print("Trabajando en:", os.getcwd())

Trabajando en: /content/drive/MyDrive/taller_calidad


Ahora, tenemos este estructura de folder in Google Drive:  
```
/content/drive/MyDrive/taller_calidad/
│
└──
```

Para poder hacer commits, Git necesita conocer la identidad del autor de cada commit.
Aqui vamos a configurar con un email y un nombre solamente para el ejercicio.
<br>

Un detalle importante: **tu `user.name` y tu `user.email` no son las credenciales de tu cuenta de GitHub**. Simplemente son datos que Git asocia a tus commits.


In [4]:
!git config --global user.email "mi-correo@example.com"
!git config --global user.name "Camille"

# Verificamos que haya quedado configurado
!git config user.name
!git config user.email

Camille
mi-correo@example.com


## Parte 1: Por que estructurar un proyecto de datos

Cuando un analisis vive en un solo notebook gigante, mezclando carga de datos, limpieza, graficos y conclusiones, es dificil de mantener y casi imposible de reutilizar. Los dos problemas mas comunes son:

- **Errores de rutas**: el notebook asume que un archivo esta en cierta carpeta, y deja de funcionar en la maquina de otra persona.
- **Dependencias faltantes**: el notebook usa una libreria que nadie mas tiene instalada, o una version distinta que cambia el resultado.

La solucion: separar responsabilidades en carpetas claras, y declarar explicitamente que librerias usa el proyecto. Un estandar muy usado en la industria (basado en Cookiecutter Data Science) se ve asi:

```
taller_calidad/
│
├── data/              # Datos crudos y procesados
├── src/               # Codigo fuente modular (.py)
├── tests/             # Pruebas unitarias
├── requirements.txt   # Dependencias del proyecto
├── .gitignore         # Archivos que Git debe ignorar
└── README.md          # Documentacion principal
```

Vamos a crear exactamente esta estructura.

Creamos las tres carpetas principales del proyecto.

In [5]:
os.makedirs("data", exist_ok=True) # Creamos la carpeta data si no existe; exist_ok=True evita un error si ya existe
os.makedirs("src", exist_ok=True)
os.makedirs("tests", exist_ok=True)

Verifiquemos que las carpetas quedaron donde esperamos. Usamos `find`, que ya viene disponible y nos da una vista clara de la estructura sin depender de nada externo.

In [6]:
!find . -not -path './.git*' -not -path '.'  # Mostramos la estructura del proyecto, excluyendo la carpeta interna de Git
# -not -path './.git*' → excluye todo lo que pertenezca a .git, incluyendo su contenido
# -not -path '.' → evita mostrar el propio directorio actual.

./data
./src
./tests


## Parte 2: `requirements.txt` - haciendo el entorno reproducible

Si un companero quiere correr tu analisis, necesita saber exactamente que librerias usar, y en que version. Sin eso, es facil que el codigo falle por una diferencia de version entre pandas 1.5 y pandas 2.1, por ejemplo.

El archivo `requirements.txt` resuelve esto: es una lista simple de librerias (y opcionalmente sus versiones) que cualquier persona puede instalar con un solo comando.

Vamos a crear el archivo con las dos librerias que vamos a necesitar esta semana: `pandas` para el procesamiento de datos, y `pytest` para las pruebas unitarias que escribiremos en la sesion 3.

In [7]:
# "%%writefile requirements.txt"  es una magic command de Colab que guarda el contenido de esta celda en el archivo indicado
%%writefile requirements.txt
pandas  # Análisis y manipulación de datos
pytest  # Pruebas unitarias

Writing requirements.txt


Con este archivo, cualquier persona (o nosotros mismos en otra maquina) podria preparar el entorno con:

```
pip install -r requirements.txt
```
Es equivalente a decir “Install all the packages listed in requirements.txt.”


In [8]:
!pip install -r requirements.txt

## Parte 3: `.gitignore` - lo que Git no debe guardar

No todo lo que hay en una carpeta de proyecto deberia subirse a un repositorio. Hay dos razones principales para excluir archivos:

- **Peso**: archivos de datos grandes (`.csv`, `.parquet`) inflan el repositorio innecesariamente. Los datos crudos generalmente se comparten por otro medio, no por Git.
- **Seguridad**: archivos generados automaticamente por Python (como las carpetas `__pycache__/`) no aportan nada al historial, y archivos con credenciales o contraseñas nunca deben terminar en un repositorio, mucho menos en uno publico.

El archivo `.gitignore` le dice a Git que patrones de archivos debe ignorar por completo, para que ni siquiera aparezcan como pendientes de subir.

Vamos a crear el file ".gitignore" con los siguientes elementos:

- `__pycache__/` → ignora la carpeta de archivos temporales de Python (Python genera archivos automáticamente, y no los necesitamos).
- `*.csv` → ignora todos los archivos CSV .
- `*.parquet` → ignora todos los archivos Parquet.
- `.ipynb_checkpoints/` → ignora los archivos temporales de checkpoints de Jupyter/Colab.   

Todos esos archivos serán ignorados en cualquier parte del repositorio.

In [9]:
%%writefile .gitignore
__pycache__/
*.csv
*.parquet
.ipynb_checkpoints/

Writing .gitignore


In [10]:
# verificamos los archivos que ahora existen en el repositorio ("-a" para ver archivos ocultos tambien)
!ls -a

.  ..  data  .gitignore  requirements.txt  src	tests


## Parte 4: `README.md`

El `README.md` es lo primero que alguien lee cuando llega a un repositorio. No hace falta que sea extenso, pero si debe explicar de que trata el proyecto y como usarlo.

In [11]:
%%writefile README.md
# Taller de Calidad

Proyecto de practica del Bootcamp Python PY4, modulo Big Data y MLOps.

## Objetivo

Aplicar buenas practicas de ingenieria de software a un analisis de datos:
control de versiones, documentacion y pruebas unitarias.

## Estructura

- `data/`: datos de entrada.
- `src/`: codigo fuente del procesamiento.
- `tests/`: pruebas unitarias con pytest.

## Como instalar dependencias

```
pip install -r requirements.txt
```

Writing README.md


## Parte 5: Inicializando el repositorio Git

Ya tenemos la estructura del proyecto lista. Ahora la convertimos en un repositorio Git, para que cada cambio que hagamos de aqui en adelante quede registrado en un historial.

`git init` crea una carpeta oculta `.git` dentro del proyecto. Esa carpeta es la que contiene todo el historial de versiones. Si se borra, se pierde el historial (aunque los archivos actuales seguirian ahi).  


In [12]:
!git init # creamos una carpeta oculta llamada `.git` dentro del proyecto.

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/drive/MyDrive/taller_calidad/.git/


**La carpeta `.git`**

La carpeta `.git` que acabamos de creer con `git init` contiene toda la información interna que Git necesita para gestionar el repositorio, como el historial de commits, las ramas, la configuración y los objetos del repositorio.

>Parra ver la carpeta en su machina (si tiene Google Drive instalado): por defecto, los sistemas operativos ocultan las carpetas cuyo nombre comienza con `.`.
>
> Mostrar archivos ocultos:  
>
>**macOS — Finder**: ⌘ Command + Shift + .`
>
>**Windows — Explorador de archivos**:
>1. Abre la carpeta del proyecto.
>2. Selecciona **Ver → Mostrar → Elementos ocultos**.

**¿Qué hay dentro de `.git`?**

- `config`: configuración específica de este repositorio.
- `HEAD`: indica cuál es la rama actualmente activa.
- `objects/`: almacena los objetos internos que Git utiliza para representar los datos y el historial.
- `refs/`: contiene referencias a ramas y otras referencias del repositorio.

No es necesario modificar estos archivos manualmente. **Es mejor dejar que Git los gestione mediante sus comandos**, como `git add`, `git commit`, `git branch` y `git config`.

> **Importante:** la carpeta `.git` es la que contiene la información que convierte nuestra carpeta de proyecto en un repositorio de Git. Si la eliminamos, los archivos del proyecto seguirán existiendo, pero perderemos el repositorio y su historial local de Git.

**Sobre el entorno de Colab**:  
El explorador de archivos de Colab (el panel de la izquierda) tambien oculta por defecto los archivos y carpetas que empiezan con un punto, como `.git` o `.gitignore`. No significa que no existan, solo que Colab no los muestra ahi. A lo largo del notebook vamos a usar `ls -la` para verlos directamente.

Confirmemos que la carpeta `.git` realmente existe, usando `ls -la` para revelar archivos ocultos.

In [13]:
!ls -la

total 26
drwx------  6 root root 4096 Sep 23 23:48 .
drwx------ 12 root root 4096 Sep 23 23:47 ..
drwx------  2 root root 4096 Sep 23 23:48 data
drwx------  7 root root 4096 Sep 23 23:48 .git
-rw-------  1 root root   49 Sep 23 23:48 .gitignore
-rw-------  1 root root  440 Sep 23 23:48 README.md
-rw-------  1 root root   73 Sep 23 23:48 requirements.txt
drwx------  2 root root 4096 Sep 23 23:48 src
drwx------  2 root root 4096 Sep 23 23:48 tests


`git status` es el comando que vamos a usar constantemente. Nos dice en que estado esta el repositorio: que archivos son nuevos, cuales fueron modificados, y cuales todavia no estan siendo rastreados por Git.

En este momento, todos nuestros archivos son nuevos para Git, asi que deberian aparecer como "untracked".

In [14]:
!git status

On branch master

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore
	README.md
	requirements.txt

nothing added to commit but untracked files present (use "git add" to track)


El archivo `.gitignore` aparece como `untracked` porque queremos que Git también registre este archivo. Su función es indicar qué otros archivos deben ser ignorados.

Por ejemplo, si existiera un archivo `.csv` en el proyecto, Git no lo mostraría en `git status` porque `*.csv` está definido en `.gitignore`.

Ahora vamos a preparar estos archivos para el commit. Este paso se llama **staging** y se realiza con `git add`.

Usar `git add .` agrega al área de staging los archivos de la carpeta actual que no estén excluidos por `.gitignore`.

In [15]:
!git add .

Si corremos `git status` de nuevo, deberiamos ver los mismos archivos, pero ahora en verde (o bajo la seccion "Changes to be committed"), indicando que estan listos para el commit.

In [16]:
!git status

On branch master

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .gitignore
	new file:   README.md
	new file:   requirements.txt



Ahora si, hacemos el commit. Un commit es una "fotografia" del estado del proyecto en un momento dado, acompañada de un mensaje que explica que se hizo. Un buen mensaje de commit es corto, claro, y describe la intencion del cambio.

In [17]:
!git commit -m "Estructura inicial del proyecto: carpetas, requirements y gitignore"

[master (root-commit) 53883aa] Estructura inicial del proyecto: carpetas, requirements y gitignore
 3 files changed, 26 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 README.md
 create mode 100644 requirements.txt


`git log` nos muestra el historial de commits. Por ahora deberiamos ver uno solo.

In [18]:
!git log --oneline

53883aa (HEAD -> master) Estructura inicial del proyecto: carpetas, requirements y gitignore


### Hagamos un segundo commit

En un proyecto real, no se hace un solo commit gigante al inicio y ya. Se hacen commits pequeños y frecuentes, cada uno representando un paso logico. Vamos a practicar esto creando el archivo donde trabajaremos la logica del proyecto en la sesion 2, y dejandolo listo con un commit propio.

Por ahora, el archivo va a estar vacio (o con un comentario placeholder). En la proxima sesion le agregaremos la funcion `calcular_metricas_ventas` con su documentacion.

In [19]:
%%writefile src/procesamiento.py
# Aqui vamos a desarrollar la logica de procesamiento de datos.
# Sesion 2: agregaremos la funcion calcular_metricas_ventas con su docstring.

Writing src/procesamiento.py


Repetimos el flujo que ya conocemos: `git status` para ver el cambio, `git add` para prepararlo, y `git commit` para guardarlo en el historial.

In [20]:
!git status

On branch master
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	src/

nothing added to commit but untracked files present (use "git add" to track)


In [21]:
!git add .
!git commit -m "Agrega archivo inicial src/procesamiento.py"

[master 65f550c] Agrega archivo inicial src/procesamiento.py
 1 file changed, 2 insertions(+)
 create mode 100644 src/procesamiento.py


Revisamos el historial de nuevo. Ahora deberian aparecer dos commits.

In [22]:
!git log --oneline

65f550c (HEAD -> master) Agrega archivo inicial src/procesamiento.py
53883aa Estructura inicial del proyecto: carpetas, requirements y gitignore


- `61082a4` → identificador del commit.
- `HEAD -> master` → estamos en la rama `master` y este es el commit actual.
- `Agrega archivo inicial src/procesamiento.py` → describe el cambio realizado.

> **Nota sobre la branch master (rama principal):**  
> Una branch (rama) es una línea de trabajo independiente que permite desarrollar cambios sin afectar directamente otras ramas. 'master' es  el nombre de la rama inicial que Git creó (a vecez se llama 'main').

## Verificacion: comprobando que el `.gitignore` funciona

Configuramos el `.gitignore` para que ignore archivos `.csv`, pero todavia no lo hemos probado. Vamos a crear un archivo `.csv` de prueba dentro de `data/` y confirmar que Git lo ignora por completo.

In [23]:
import pandas as pd

df_prueba = pd.DataFrame({"producto": ["A", "B"], "cantidad": [10, 5]})
df_prueba.to_csv("data/ventas_prueba.csv", index=False)

Ahora revisamos `git status`. Si el `.gitignore` esta funcionando correctamente, este archivo `.csv` **no** deberia aparecer en la lista de archivos nuevos.

In [24]:
!git status

On branch master
nothing to commit, working tree clean


Si el archivo `.csv` no aparece en la lista, el `.gitignore` esta cumpliendo su funcion: los datos pueden existir en la carpeta del proyecto sin terminar en el historial de Git.

## Cierre de la sesion

Repasemos lo que construimos hoy:

- Una estructura de proyecto con `data/`, `src/` y `tests/`.
- Un `requirements.txt` que declara las dependencias del proyecto.
- Un `.gitignore` que evita subir archivos pesados o generados automaticamente, y ya lo probamos con un `.csv`.
- Un repositorio Git inicializado, con dos commits en su historial.

Este mismo proyecto lo vamos a retomar en la sesion 2. Como esta guardado en Drive, en la proxima sesion solo necesitamos montar Drive de nuevo y ubicarnos en la misma carpeta con `os.chdir(PROJECT_PATH)` para seguir exactamente donde quedamos, historial de Git incluido.

En la sesion 2 vamos a desarrollar la funcion `calcular_metricas_ventas` dentro de `src/procesamiento.py`, documentandola siguiendo el estandar de docstrings estilo NumPy.

### Antes de terminar, verifica que puedes responder:

- Por que separar `data/`, `src/` y `tests/` en carpetas distintas en vez de tener todo junto.
- Que diferencia hay entre `git add` y `git commit`.
- Por que un archivo `.csv` no deberia subirse normalmente a un repositorio Git.